In [5]:
import io
import pickle
from pathlib import Path
import pickletools

In [8]:
def load_safe_partial_dict(file_path):
    with open(file_path, 'rb') as f:
        data = f.read()

    stream = io.BytesIO(data)
    unpickler = pickle.Unpickler(stream)

    try:
        obj = unpickler.load()
        print("Loaded full object successfully.")
        return obj
    except Exception:
        print("File appears truncated. Attempting partial recovery...")

    # Try to recover individual entries manually
    stream.seek(0)
    partial_recovered = {}

    try:
        # Read past initial pickle setup
        header_unpickler = pickle.Unpickler(stream)
        stack = header_unpickler.load()  # Tries to load the dict directly
    except Exception:
        pass  # We'll retry below using a lower-level scan

    # Attempt low-level scan of key-value pairs
    stream.seek(0)
    while stream.tell() < len(data):
        try:
            key_pos = stream.tell()
            key = pickle.load(stream)

            if not isinstance(key, str):
                print(f"Invalid key at byte {key_pos}, skipping...")
                continue

            value_pos = stream.tell()
            value = pickle.load(stream)

            if not isinstance(value, list):
                print(f"Invalid value at byte {value_pos}, skipping key '{key}'...")
                continue

            partial_recovered[key] = value

        except Exception as e:
            print(f"\nStopped at byte {stream.tell()}: {e}")
            break

    print(f"\nRecovered {len(partial_recovered)} key-value pairs.")
    return partial_recovered


def recover_pairs_from_pickle(data):
    recovered = {}
    raw_items = []

    # Step 1: Try to find all memoized key-value chunks
    try:
        ops = list(pickletools.genops(data))
    except Exception as e:
        print(f"Error parsing pickle stream: {e}")
        return recovered

    # Step 2: Look for GET or BINPUT storing dict items
    stack = []
    current_key = None
    last_pos = 0

    for op, arg, pos in ops:
        if op.name in {"SETITEM", "SETITEMS"}:
            # Use everything from last_pos to current op as a chunk
            chunk_end = pos
            chunk_bytes = data[last_pos:chunk_end]
            raw_items.append(chunk_bytes)
            last_pos = chunk_end
        elif op.name == "STOP":
            break

    # Step 3: Try to load each chunk independently
    for idx, raw in enumerate(raw_items):
        try:
            # Wrap the chunk as a dict for loading
            wrapped = b'\x80\x04}\x94' + raw + b'.'
            obj = pickle.loads(wrapped)
            recovered.update(obj)
        except Exception:
            continue  # Skip bad chunks

    return recovered

def recover_pickle_dict(path):
    with open(path, 'rb') as f:
        data = f.read()

    print(f"Read {len(data)} bytes.")
    print("Scanning for recoverable entries...")

    partial = recover_pairs_from_pickle(data)
    print(f"Recovered {len(partial)} intact key-value pairs.")
    return partial


def recover_dict_from_truncated_pickle(file_path):
    recovered = {}
    with open(file_path, "rb") as f:
        raw = f.read()

    stream = io.BytesIO(raw)
    unpickler = pickle.Unpickler(stream)

    print("Trying to load entries one-by-one...")

    while True:
        try:
            # Read a key (must be str)
            pos_before_key = stream.tell()
            key = unpickler.load()
            if not isinstance(key, str):
                print(f"Non-string key at {pos_before_key}, skipping...")
                continue

            # Read a value (must be list)
            pos_before_val = stream.tell()
            value = unpickler.load()
            if not isinstance(value, list):
                print(f"Non-list value at {pos_before_val} for key '{key}', skipping...")
                continue

            recovered[key] = value
        except EOFError:
            print("Reached end of file.")
            break
        except Exception as e:
            print(f"Stopped at byte {stream.tell()}: {e}")
            break

    print(f"✅ Recovered {len(recovered)} key-value pairs.")
    return recovered

In [3]:
# Set target file
file_path = Path.home() / "Desktop" / "eval_runner_simple_responses_temp.pkl"
recovered_file_path = Path.home() / "Desktop" / "recovered.pkl"

In [9]:
# Read the partial pickle file
#recovered_data = load_safe_partial_dict(file_path) # V1
#recovered_data = recover_pickle_dict(file_path) # V2
recovered_data = recover_dict_from_truncated_pickle(file_path) # V3

Trying to load entries one-by-one...
Stopped at byte 821169404: pickle data was truncated
✅ Recovered 0 key-value pairs.


In [ ]:
# Save the recovered data to a new pickle file
with open(recovered_file_path, "wb") as f:
    pickle.dump(recovered_data, f)